# Waldbrände - Vitalität der Vegetation berechnen und Veränderungen erkennen

In der vorherigen Übung habt ihr gelernt, dass unterschiedliche Materialien die verschiedenen Wellenlängenbereiche bzw. Farben des Lichts unterschiedlich stark reflektieren. Vegetation reflektiert im sichtbaren Bereich eine Mischung, die für das Auge grün erscheint, und zusätzlich auch Infrarotlicht. Gestresste Vegetation dagegen reflektiert weniger Infrarot. Andere grün erscheinende Materialien wie Kunstrasen reflektieren gar kein Infrarot.

In diesem Teil lernt ihr eine Methode, mit der die Vitalität der Vegetation am Beispiel eines Waldbrandes in Griechenland untersucht werden kann. Dafür visualisiert und analysiert ihr die Veränderungen der Vegetation über einen vorgegebenen Zeitraum.

# 1 Pakete importieren

In [1]:
import folium                                   # interaktive Karte erzeugen (einfachere Version)
import matplotlib.pyplot as plt                 # Für Diagramme und Visualisierungen
import numpy as np                              # Für numerische Operationen mit Arrays (Datacubes)
import os                                       # Für Betriebssystem-Funktionen (z.B. Dateipfade)
import rasterio                                 # Zum Lesen/Schreiben von GeoTIFF-Dateien

from geolibre import Map                        # Für Geodaten-Visualisierung auf Karten

# 2 Waldbrand in Griechenland

## 2.1 Vitalität der Vegetation untersuchen

Um herauszufinden, wie gesund die Vegetation in den Satellitenbildern ist, wird der **NDVI** (Normalized Difference Vegetation Index) verwendet. Der NDVI ist ein Index, der die Reflektion des roten Lichts mit der des infraroten Lichts vergleicht bzw. verrechnet. Durch eine relativ einfache mathematische Formel entstehen **Werte zwischen -1 und +1**. Je höher der NDVI eines Pixels ist, desto vitaler (gesünder) ist die Vegetation, die abgebildet wird – wenn überhaupt. Der Wert 0 bedeutet keine oder tote Vegetation, und der Wert 1 steht für extrem grüne und vitale Vegetation (weitere Infos). Diese Werte können dann mithilfe beliebiger Farbskalen dargestellt werden.

Wenn der NDVI zu verschiedenen Zeitpunkten berechnet und verglichen wird, kann nicht nur gesunde von gestresster Vegetation unterschieden, sondern auch Veränderungen der Vegetation zuverlässig untersucht werden. Es gibt viele sinnvolle Anwendungsfälle, wie beispielsweise die Bestimmung von Brandflächen nach Waldbränden oder das Erfassen von Rodungen in Wäldern. Anders herum lässt sich auch das Nachwachsen von Vegetation nach solchen Einschnitten dokumentieren.

Zuerst braucht ihr einen Untersuchungsraum. Dieses Mal begebt ihr euch nach Griechenland! Die folgende Codezelle speichert die Koordinaten in der Variable `aoi`. Mit der Codezelle danach könnt ihr eure AOI (area of interest) auf der Karte sehen.

In [ ]:
# Wo (Evros, Griechenland)
aoi = {"west": 25.73, "south": 40.86, "east": 25.93, "north": 40.97}

In [ ]:
# Kontrolle der Koordinaten
bounds = [[aoi['south'], aoi['west']], [aoi['north'], aoi['east']]]
m = folium.Map(tiles='OpenStreetMap', control_scale=True)
folium.Rectangle(bounds=bounds, color='red', weight=2, fill=True, fill_opacity=0.1).add_to(m)
m.fit_bounds(bounds)
display(m)

Für diese Untersuchung werden bereits vorgefertigte NDVI-Bilder von dem Datensatz [Terrascope](https://stacbrowser.terrascope.be/collections/terrascope-s2-ndvi-v2) verwendet.

Um die Veränderung der Vegetation nachzuvollziehen, werden drei Aufnahmezeitpunkte miteinander verglichen. Einmal vor dem Waldbrand (Juni/Juli 2023), während des Brandes (August/September 2023) und nach dem Brand (September/November 2023).

In der folgenden Codezeile werden die Aufnahmen mit dem Paket `Matplotlib` visualisiert.

:::{.task}

#### Aufgabe 1:

Ändert den Titel und die Achsen-Beschriftung für alle drei Abbildungen. Die relevanten Zeilen findet ihr im Code kommentiert.

:::

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

files = [
    ("./data/ndvi-greece-pre-2023_cog.tif", "Pre-Event"),           # An der Stelle von "Pre-Event" könnt ihr den Titel der Abbildung ändern
    ("./data/ndvi-greece-nrt-2023_cog.tif", "Near Real-Time"),      # An der Stelle von "Near Real-Time" könnt ihr den Titel der Abbildung ändern
    ("./data/ndvi-greece-post-2023_cog.tif", "Post-Event")          # An der Stelle von "Post-Event" könnt ihr den Titel der Abbildung ändern
]

for ax, (file, title) in zip(axes, files):
    if os.path.exists(file):
        with rasterio.open(file) as src:
            img = ax.imshow(src.read(1), cmap='RdYlGn', vmin=-0.1, vmax=0.9)    # Wertebereich für die Farbskala festlegen

            ax.set_title(title)
            ax.set_xlabel("Pixel X")                                          # Bei "Pixel X" könnt ihr die Achse beschriften
            ax.set_ylabel("Pixel Y")                                          # Bei "Pixel Y" könnt ihr die Achse beschriften
            fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04, label="NDVI")  # Bei "NDVI" könnt ihr die Beschriftung der Farbskala ändern
    else:
        ax.set_title(f"{title} (Missing)")
        ax.axis('off')

plt.tight_layout()
plt.show()
# plt.savefig("ndvi")

## 2.2 Die NDVI-Entwicklung darstellen

Die Veränderung der Vegetation lässt sich nicht nur in drei Bildern nebeneinander abbilden, sondern auch in einem Differenzbild darstellen. Dafür werden die Werte vor dem Waldbrand von den Werten nach dem Brand abgezogen. Dadurch erhalten abnehmende Werte, also Stellen mit einem schlechteren NDVI Wert nach dem Brand, ein negatives Vorzeichen.

In [ ]:
pre_file = "./data/ndvi-greece-pre-2023_cog.tif"
post_file = "./data/ndvi-greece-post-2023_cog.tif"
output_diff_file = "./data/ndvi-greece-diff-2023_cog.tif"

if os.path.exists(pre_file) and os.path.exists(post_file):
    with rasterio.open(pre_file) as pre_src, rasterio.open(post_file) as post_src:
        pre_ndvi = pre_src.read(1).astype('float32')
        post_ndvi = post_src.read(1).astype('float32')

        # Differenz berechnen
        diff_ndvi = post_ndvi - pre_ndvi

        # Metadaten für Export extrahieren
        profile = pre_src.profile.copy()
        extent = [pre_src.bounds.left, pre_src.bounds.right, pre_src.bounds.bottom, pre_src.bounds.top]

    # Speichern als GeoTIFF
    profile.update(dtype=rasterio.float32, count=1, driver='GTiff')
    with rasterio.open(output_diff_file, 'w', **profile) as dst:
        dst.write(diff_ndvi.astype(rasterio.float32), 1)
    print(f"Datei gespeichert: {output_diff_file}")

    # Visualisierung
    plt.figure(figsize=(12, 8))
    img = plt.imshow(diff_ndvi, cmap='RdBu', vmin=-0.5, vmax=0.5, extent=extent)
    plt.title("NDVI Differenz")                         # Hier kann der Titel der Grafik umbenannt werden
    plt.xlabel("Longitude / Längengrad")                # Hier kann die x-Achse umbeschriftet werden
    plt.ylabel("Latitude / Breitengrad")                # Hier kann die y-Achse umbeschriftet werden
    plt.colorbar(img, label="NDVI Veränderung")         # Hier kann die Legende rechts umbeschriftet werden
    plt.show()
else:
    print("Dateien für Differenzberechnung nicht gefunden.")
    # plt.savefig("ndvi-diff")

:::{.task}

#### Aufgabe 2

Beschreibt die Abbildung und erläutert, was es mit der NDVI-Veränderung auf sich hat. Weshalb seht ihr nur eine Veränderung im roten Bereich?

*(10 Minuten)*
:::

:::{.content-visible when-format="html"}
:::{.callout-warning collapse=true}

#### Lösung

Die Abbildung zeigt die NDVI-Veränderung nach dem Waldbrand verglichen mit der Zeit vor dem Waldbrand. Also wie sich der NDVI durch den Waldbrand verändert bzw. entwickelt hat. Man sieht nur eine Veränderung im roten minus-Bereich, weil der NDVI durch den Waldbrand gesunken ist. Das heißt, die Vegetation hat an Vitalität verloren und ist stark gestresst oder sogar komplett verbrannt.

:::
:::

## 2.3 Pixel-Inspektor

Mit dem Pixel-Inspektor Tool könnt ihr euch für jeden Pixel seinen NDVI-Wert anzeigen lassen. Wenn ihr die nachfolgende Codezelle ausführt, erscheint eine Karte mit den oberen vier Satellitenbildern (vor dem Brand, während des Brandes, nach dem Brand sowie das Differenzbild mit der NDVI-Veränderung).

Mit den Reglern könnt ihr die Bilder jeweils ein- und ausblenden (wie bereits in Teil 2). Die Bilder sind zu Beginn Schwarz-Weiß. Mit einem Klick auf die Farbpalette ![Geolibre Farbpalette Symbol](images/CDEC_M1_Teil3_Farbpalette.png) könnt ihr jedes Bild unter *Farbverlauf* farbig visualisieren. Wenn ihr zum Beispiel beim Vorher-Bild die Farben ändern wollt, klickt ihr dort auf das Farbpaletten-Symbol. Mit einem Klick auf *Verlauf umkehren* kann der Farbverlauf auch umgekehrt werden. Nun könnt ihr die Bilder besser miteinander vergleichen.

:::{.callout-note}
#### Hinweis

Wenn das Differenz-Bild komplett schwarz ist, muss der Wertebereich angepasst werden. Dafür klickt ihr wieder auf die Farbpalette und gebt bei *Min* den Wert -0.7 und bei *Max* 0.01 ein.

:::

Um jetzt den NDVI-Wert eines Pixels abzulesen, klickt ihr auf das Inspektor-Symbol (siehe Screenshot) links neben der Farbpalette. Wenn ihr euch zum Beispiel für das Nachher-Bild die Werte abrufen wollt, klickt ihr dort auf das Inspektor-Symbol. Danach könnt ihr auf einen beliebigen Bereich auf dem Bild klicken und ihr könnt den NDVI-Wert für den Pixel abrufen.

![Geolibre Pixel-Inspektor Symbol](images/CDEC_M1_Teil3_pixel-inspector.png)

In [ ]:
# Karte mit Geolibre visualisieren

m = Map(center=[25.824420, 40.918085], zoom=11)    # [Lon, Lat]

m.add_tile_layer(
    "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="ESRI World Imagery")

diff = m.add_cog("./data/ndvi-greece-diff-2023_cog.tif", name="Veränderung")
post = m.add_cog("./data/ndvi-greece-post-2023_cog.tif", name="Nachher")
nrt = m.add_cog("./data/ndvi-greece-nrt-2023_cog.tif", name="Währenddessen")
pre = m.add_cog("./data/ndvi-greece-pre-2023_cog.tif", name="Vorher")

m

:::{.task}

#### Aufgabe 3

Vergleicht mit dem Pixel-Inspektor die NDVI-Werte der verschiedenen Aufnahmezeitpunkte und untersucht, welche Bereiche sich besonders stark und welche sich fast gar nicht verändert haben.

*(10 Minuten)*
:::

:::{.content-visible when-format="html"}
:::{.callout-warning collapse=true}

#### Lösung

Vor allem Bereiche in der Stadt Alexandroupolis zeigen (glücklicherweise) kaum NDVI-Veränderungen, während Bereiche in den Waldgebieten nordwestlich teilweise komplett abgebrannt sind.

:::
:::

## 2.4 Histogramme

Die Entwicklung des NDVI lässt sich auch anders darstellen und vergleichen. Hier werden alle Pixel im Untersuchungsgebiet gesammelt (Y-Achse) und nach ihrem Wert (X-Achse) sortiert. Es sind z.B. knapp 40.000 Pixel mit einem NDVI-Wert von 0,4 in dem Untersuchungsgebiet vor dem Brand.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharey=True)    # Y-Achse für beide Grafiken gleich

if os.path.exists(pre_file) and os.path.exists(post_file):
    # Histogramm für Pre-Event
    axes[0].hist(pre_ndvi.flatten(), bins=50, color='green', alpha=0.7)
    axes[0].set_title("NDVI Verteilung: Vor dem Brand")
    axes[0].set_xlabel("NDVI Wert")
    axes[0].set_ylabel("Häufigkeit")

    # Histogramm für Post-Event
    axes[1].hist(post_ndvi.flatten(), bins=50, color='red', alpha=0.7)
    axes[1].set_title("NDVI Verteilung: Nach dem Brand")
    axes[1].set_xlabel("NDVI Wert")

plt.tight_layout()
plt.show()
# plt.savefig("histogram")

Die nächste Grafik vereint beide Grafiken von oben. Damit kann die Veränderung noch auf eine andere Weise dargestellt werden, enthält aber sogesehen die gleichen Informationen wie oben. Die gestrichelte vertikale Linie ist der Median des jeweiligen Zeitpunkts.

In [ ]:
plt.figure(figsize=(12, 7))

if 'pre_ndvi' in locals() and 'post_ndvi' in locals():
    # Überlagertes Histogramm
    plt.hist(pre_ndvi.flatten(), bins=50, color='green', alpha=0.5, label='Vorher')
    plt.hist(post_ndvi.flatten(), bins=50, color='red', alpha=0.5, label='Nachher')

    plt.title("Vergleich der NDVI-Verteilung: Vorher - Nachher")
    plt.xlabel("NDVI Wert")
    plt.ylabel("Häufigkeit (Pixelanzahl)")
    plt.legend(loc='upper right')
    plt.grid(axis='y', linestyle='--', alpha=0.3)

    # Median-Linien einzeichnen für bessere Analyse
    plt.axvline(np.nanmedian(pre_ndvi), color='darkgreen', linestyle='dashed', linewidth=2, label='Median Pre')
    plt.axvline(np.nanmedian(post_ndvi), color='darkred', linestyle='dashed', linewidth=2, label='Median Post')

    plt.show()
else:
    print("NDVI-Daten wurden nicht gefunden. Bitte führe die vorherigen Zellen erneut aus.")
    # plt.savefig("ndvi-histogram-layered")

:::{.task}

#### Aufgabe 4

Vergleicht den NDVI vor dem Brand mit dem NDVI nach dem Brand. Wie haben sich die NDVI-Werte durch das Ereignis verändert und welche Rückschlüsse könnt ihr daraus ziehen?

*(10 Minuten)*
:::

:::{.content-visible when-format="html"}
:::{.callout-warning collapse=true}

#### Lösung

Vor dem Waldbrand lag der NDVI zwischen 0,1 und 0,9. Die Vegetation ist zu dem Zeitpunkt also noch sehr gesund und teilweise sehr dicht. In den Bereichen 0,7 bis 0,9 ist ein teils starker Anstieg an Häufigkeit der Pixel für diese Werte. Das heißt, sehr viele Pixel liegen in diesem Wertbereich. Zum Beispiel haben über 110.000 Pixel einen NDVI-Wert von 0,8 und unter 40.000 Pixel einen NDVI-Wert von 0,2.

Nach dem Waldbrand dagegen liegen die meisten Werte zwischen 0 und 0,4. Zum Beispiel haben über 210.000 Pixel einen NDVI-Wert von 0,2 und unter 10.000 Pixel nur noch einen NDVI-Wert von 0,8. Die Vegetation ist gestresst und nicht mehr so dicht wie vorher. Die betroffene Landschaft ist vermutlich eher karg geworden.

:::
:::

# 3 Zusammenfassung

In diesem Notebook habt ihr gelernt, wie ihr:

- Aussagen über den Zustand der Vegetation im Satellitenbild treffen könnt (NDVI)
- die Vitalität der Vegetation vergleicht und Veränderungen durch einen Waldbrand deutet
- mit `Matplotlib` Grafiken erstellt und diese anpasst
- eine NDVI-Differenzkarte erstellt und interpretiert
- mit dem Inspektor Tool Pixelwerte abfragt